# Step 2 — Day 2: Source-Specific NLP Cleaning

**Author:** Khelil Dhiaeddine  
**Project:** Sentiment Analysis for Financial Market Prediction (TSLA, 2020–2023)  
**Supervisor:** Nesrine Lahiani  

---

## Context

In Day 1, I ran FinBERT on 200 raw rows (50 per source) **without any cleaning** to understand what the model actually struggles with before writing a single line of preprocessing code.

Here is what I found:

| Source | Critical Issues | Low-Confidence Preds |
|---|---|---|
| news | Newlines 35/50, Short texts 11/50 | 0/50 ✅ |
| reddit | Newlines 25/50, URLs 8/50, Markdown 5/50 | 8/50 ⚠️ |
| twitter_general | Cashtags 36/50, URLs 24/50, Mentions 10/50, Emojis 11/50 | 3/50 |
| twitter_musk | Mentions **42/50**, Short texts 9/50 | 2/50 |

**Key observation:** All sources showed ~70–84% neutral predictions on raw text. That is FinBERT defaulting to neutral when it cannot parse noisy input — not genuine neutrality. My hypothesis is that cleaning will redistribute labels away from neutral significantly, especially for twitter_musk (42/50 mentions eating signal).

**This notebook:** Implements source-specific cleaning functions derived from Day 1 evidence, validates them, and produces a clean Layer 2 dataset ready for full FinBERT inference in Day 4.

---

## Design Decisions (made before writing any code)

1. **Cashtags (`$TSLA`):** Keep as normalized token (`TSLA`) — not stripped. Cashtags are a positive signal that the text is explicitly about Tesla. Stripping them loses information.
2. **Mentions (`@elonmusk`):** Strip entirely. They carry no sentiment signal for financial prediction.
3. **Newlines:** Normalize to single space. FinBERT was trained on continuous prose.
4. **Short texts (<5 words after cleaning):** Flag with `is_too_short=True` but do NOT drop at this stage — dropping happens at the feature engineering step with full justification.
5. **Sarcasm (Reddit):** Cannot be fixed by regex. Document as a known limitation. Low-confidence predictions on Reddit are expected to persist post-cleaning.
6. **One cleaning function per source:** Each source has a distinct noise profile. A single universal cleaner would be academically indefensible.

---
## 0. Setup

In [ ]:
import os
import sys
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Allow src/ imports without pip install -e .
sys.path.insert(0, os.path.abspath('..'))

from src.nlp.cleaning import (
    clean_text,
    flag_short_text,
    CLEANER_MAP,
    # individual utilities — available if needed for inspection
    remove_urls, remove_mentions, normalize_cashtags,
    remove_emojis, normalize_whitespace, remove_repeated_chars, normalize_unicode,
)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 50)

print('Libraries loaded.')
print(f'Available cleaners: {list(CLEANER_MAP.keys())}')

---
## 1. Load Layer 2 + Day 1 Sample

I load both:
- **Full Layer 2** — what I will eventually clean entirely
- **Day 1 sample results** — I use this as my validation set. I already know what FinBERT produced on raw text, so I can measure the before/after effect of cleaning on the same 200 rows.

In [ ]:
LAYER2_PATH = os.getenv('LAYER2_PATH', '../data/raw/layer2/layer2_unified_final.parquet')
DAY1_SAMPLE = os.getenv('DAY1_SAMPLE', '../outputs/data/step2_finbert_sample_results.parquet')

df     = pd.read_parquet(LAYER2_PATH)
sample = pd.read_parquet(DAY1_SAMPLE)

print(f'Layer 2 — rows: {len(df):,} | sources: {df["source"].value_counts().to_dict()}')
print(f'Day 1 sample — rows: {len(sample)}')

---
## 2. Shared Utility Functions

These are source-agnostic helpers that all four cleaning functions call internally.
Keeping them separate from the source-specific logic makes the code testable and reusable.

In [ ]:
# All utility functions are defined in src/nlp/cleaning.py and imported in cell 0.
# This keeps the module as the single source of truth — no duplicates in the notebook.
#
# Functions available:
#   Utilities : remove_urls, remove_mentions, normalize_cashtags, remove_emojis,
#               normalize_whitespace, remove_repeated_chars, normalize_unicode,
#               flag_short_text
#   Cleaners  : clean_news, clean_reddit, clean_twitter_general, clean_twitter_musk
#   Router    : clean_text(text, source)  ← primary entry point
#
# To inspect a cleaner's logic:
#   import inspect; from src.nlp.cleaning import clean_reddit; print(inspect.getsource(clean_reddit))

print('Cleaning functions loaded from src/nlp/cleaning.py')
print(f'Cleaner map: {list(CLEANER_MAP.keys())}')

---
## 3. Source-Specific Cleaning Functions

Each source has its own function. This is intentional — the noise profiles are different enough that a universal cleaner would either under-clean Twitter or over-clean news.

---
## 4. Unit Tests — Verify Each Cleaner Before Applying to Full Data

I write explicit test cases for each cleaning decision.
This is important for the thesis: every transformation should be verifiable.

In [ ]:
test_cases = {
    'news': [
        (
            'Tesla reports record Q4 deliveries.\nShares rose 5% in after-hours trading.\nAnalysts expect further gains.',
            'Tesla reports record Q4 deliveries. Shares rose 5% in after-hours trading. Analysts expect further gains.'
        ),
        (
            'Check the full report at https://reuters.com/tesla-q4',
            'Check the full report at'
        ),
    ],
    'reddit': [
        (
            '**This is bold** and _italic_ and `code` and > a blockquote\nWith a [link](https://example.com) here',
            'This is bold and italic and code and a blockquote With a link here'
        ),
        (
            'TESLAAAAA to the mooooon 🚀🚀',
            'TESLAA to the moon'
        ),
    ],
    'twitter_general': [
        (
            'RT @elonmusk $TSLA just broke $900! 🔥 #Tesla #EV https://t.co/abc123',
            'TSLA just broke $900! Tesla EV'
        ),
        (
            '@TeslaOwners @Model3Club love my new car!! 😍😍😍',
            'love my new car!!'
        ),
    ],
    'twitter_musk': [
        (
            '@CathieDWood @ARKInvest love what you are doing with $TSLA\nhttps://t.co/xyz',
            'love what you are doing with TSLA'
        ),
        (
            'RT @sama thanks for the kind words 🙏',
            'thanks for the kind words'
        ),
    ]
}

print('=== Unit Tests ===')
all_passed = True
for source, cases in test_cases.items():
    print(f'\n--- {source} ---')
    for i, (raw, expected) in enumerate(cases):
        result = clean_text(raw, source).strip()
        passed = result == expected.strip()
        status = '✅ PASS' if passed else '❌ FAIL'
        print(f'  Test {i+1}: {status}')
        if not passed:
            all_passed = False
            print(f'    Input:    {raw[:80]}')
            print(f'    Expected: {expected[:80]}')
            print(f'    Got:      {result[:80]}')

print(f'\n{"All tests passed ✅" if all_passed else "Some tests failed ❌ — fix before continuing"}')

---
## 5. Apply Cleaning to Day 1 Sample — Before vs. After Analysis

Before cleaning the full 87K rows I validate on the 200-row sample from Day 1.
This lets me directly compare FinBERT outputs on raw vs clean text — same rows, same model.

In [ ]:
# Apply cleaning to sample
sample['text_clean'] = sample.apply(
    lambda row: clean_text(row['text'], row['source']), axis=1
)

# Flag short texts
sample['is_too_short'] = sample['text_clean'].apply(flag_short_text)

# Measure token length reduction
sample['char_len_raw']   = sample['text'].str.len()
sample['char_len_clean'] = sample['text_clean'].str.len()
sample['char_reduction'] = sample['char_len_raw'] - sample['char_len_clean']

print('=== Character Length Reduction by Source ===')
print(sample.groupby('source')[['char_len_raw', 'char_len_clean', 'char_reduction']].mean().round(1))
print()
print('=== Short Texts After Cleaning ===')
print(sample.groupby('source')['is_too_short'].sum())

In [ ]:
# ── Spot-check: side-by-side comparison of raw vs clean text ──
print('=== Raw vs Clean — 3 Samples Per Source ===')
for src in sample['source'].unique():
    subset = sample[sample['source'] == src].sample(3, random_state=42)
    print(f'\n{'='*60}')
    print(f'Source: {src}')
    print(f'{'='*60}')
    for _, row in subset.iterrows():
        print(f'  RAW  : {row["text"][:150]}')
        print(f'  CLEAN: {row["text_clean"][:150]}')
        print()

---
## 6. Re-run FinBERT on Cleaned Sample — Measure Impact

This is the key validation step. I re-run FinBERT on the same 200 rows, now cleaned.
My hypothesis: neutral predictions should drop, especially for twitter_musk.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'ProsusAI/finbert'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print(f'FinBERT loaded on {DEVICE}')
print('Label map:', model.config.id2label)

In [ ]:
import torch
from src.sentiment.finbert import load_finbert, run_finbert_batch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer, model = load_finbert(device=DEVICE)

In [ ]:
# run_finbert_batch is imported from src.sentiment.finbert — no local definition needed.

# Run on CLEAN texts — filter out too-short rows first
sample_to_run = sample[~sample['is_too_short']].copy()
print(f'Running on {len(sample_to_run)} rows (excluded {sample["is_too_short"].sum()} too-short)')

preds_clean = run_finbert_batch(sample_to_run['text_clean'].tolist(), tokenizer, model, DEVICE)
preds_clean_df = pd.DataFrame(preds_clean)
preds_clean_df.columns = [c + '_clean' for c in preds_clean_df.columns]

sample_to_run = sample_to_run.reset_index(drop=True)
sample_validated = pd.concat([sample_to_run, preds_clean_df], axis=1)

print('Done.')

In [ ]:
# ── Low-confidence comparison ──
sample_validated['max_prob_clean'] = sample_validated[['p_pos_clean', 'p_neg_clean', 'p_neu_clean']].max(axis=1)
sample_validated['low_conf_clean'] = sample_validated['max_prob_clean'] < 0.60

print('=== Low-Confidence: Before vs After ===')
print(f'{"Source":<20} {"Raw":>8} {"Clean":>8} {"Improvement":>12}')
print('-' * 52)
for src in sample_validated['source'].unique():
    sub     = sample_validated[sample_validated['source'] == src]
    raw_lc  = sub['low_confidence'].sum() if 'low_confidence' in sub.columns else 'N/A'
    cln_lc  = sub['low_conf_clean'].sum()
    improvement = raw_lc - cln_lc if isinstance(raw_lc, (int, np.integer)) else 'N/A'
    print(f'{src:<20} {str(raw_lc):>8} {cln_lc:>8} {str(improvement):>12}')

---
## 7. Apply Cleaning to Full Layer 2 Dataset

Validation passed on the 200-row sample. Now I apply the same functions to all ~87K rows.

In [ ]:
print(f'Cleaning {len(df):,} rows...')

df['text_clean'] = df.apply(
    lambda row: clean_text(row['text'], row['source']), axis=1
)

df['is_too_short'] = df['text_clean'].apply(flag_short_text)
df['text_empty']   = df['text_clean'].str.strip().eq('')

print(f'Done.')
print(f'Empty texts after cleaning: {df["text_empty"].sum()}')
print(f'Too-short texts after cleaning: {df["is_too_short"].sum()}')
print()
print('=== Too-Short by Source ===')
print(df.groupby('source')['is_too_short'].sum())

In [ ]:
# ── Schema check: Layer 2 fields preserved, text_clean added ──
EXPECTED_COLS = ['doc_id', 'published_at', 'source', 'text', 'ticker', 'url', 'text_clean', 'is_too_short']
missing = [c for c in EXPECTED_COLS if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

print('Schema check passed ✅')
print(df.dtypes)

---
## 8. Final Quality Report

A summary I can copy into my thesis methodology section and supervisor progress report.

In [ ]:
print('=' * 65)
print('STEP 2 — DAY 2 CLEANING QUALITY REPORT')
print('=' * 65)

total = len(df)
usable = df[~df['is_too_short'] & ~df['text_empty']]

print(f'\nTotal rows before cleaning : {total:>10,}')
print(f'Empty after cleaning        : {df["text_empty"].sum():>10,}')
print(f'Too short after cleaning    : {df["is_too_short"].sum():>10,}')
print(f'Usable rows for FinBERT     : {len(usable):>10,}')
print()
print('--- By Source ---')
for src, grp in df.groupby('source'):
    u = grp[~grp['is_too_short'] & ~grp['text_empty']]
    print(f'  {src:<22} total={len(grp):>6,}  usable={len(u):>6,}  '
          f'dropped={len(grp)-len(u):>5,} ({(len(grp)-len(u))/len(grp)*100:.1f}%)')

print()
print('--- Cleaning Operations Applied ---')
ops = {
    'news':            ['remove_urls', 'remove_mentions', 'normalize_whitespace'],
    'reddit':          ['remove_urls', 'strip_markdown', 'remove_emojis', 'remove_repeated_chars', 'normalize_whitespace'],
    'twitter_general': ['remove_rt_prefix', 'remove_urls', 'remove_mentions', 'normalize_cashtags', 'normalize_hashtags', 'remove_emojis', 'normalize_whitespace'],
    'twitter_musk':    ['remove_rt_prefix', 'remove_urls', 'remove_mentions', 'normalize_cashtags', 'normalize_hashtags', 'remove_emojis', 'normalize_whitespace'],
}
for src, ops_list in ops.items():
    print(f'  {src}: {" → ".join(ops_list)}')

print()
print('--- Known Limitations ---')
print('  Reddit sarcasm: cannot be resolved by rule-based cleaning.')
print('  Low-confidence predictions on Reddit may persist post-cleaning.')
print('  twitter_general 2023 coverage gap: documented, not fixable.')
print('  First-chunk truncation: information beyond token 512 is discarded.')
print('  This affects news (25/50 exceeded 512 tokens in Day 1 sample).')

---
## 9. Save Cleaned Layer 2

In [ ]:
OUTPUT_PATH = 'layer2_cleaned.parquet'

df.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Rows: {len(df):,} | Columns: {list(df.columns)}')

OUTPUT_PATH = os.path.join(
    os.getenv('OUTPUT_PATH', '../outputs'),
    'data',
    'layer2_cleaned.parquet'
)
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Rows: {len(df):,} | Columns: {list(df.columns)}')